<img src=images/gdd-logo.png width=300px align=right>

# Sklearn for Linear Regression

In this notebook you will use Scikit-Learn to perform linear regression on an E-commerce Customer Dataset.

## Goal

The goal is to predict the ‘Yearly Amount Spent’ by a customer on an E-commerce platform, so that this information can be used to give the particular customer personalized offers or Loyalty membership etc.

The notebook will help you with the initial data exploration, but you will perform the actual modelling!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## 0. Exploring the data

This is the dataset we will be performing linear regression on.

We will try to predict `Yearly Amount Spent`, so this will be our target vector (also known as dependent variable, outcome or output variable).

In [ ]:
customers = (
    pd.read_csv('data/Ecomm-Customers.csv')
    .rename(str.lower, axis='columns')
)
customers.head()

### <mark>Exercise: Perform some preliminary analysis on the dataset.</mark>

* How many customers are in this dataset?

* What datatypes does the dataset contain? Are there any missing values?

* How many predictive features are there? Which features do you think might be informative for the target?

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
Key features:
- avg. session length	
- time on app	
- time on website	
- length of membership

Notes about the other features:
- Yearly Amount Spent is of course the target so we won't use it as a predictive feature. 
- Email is unique for everyone so not going to be predictive. 
- We could maybe extract some information from the email and address (country, area etc.) to get some more predictive features, but for now let's not focus on that.
- Avatar could be predictive but there are so many different unique values we'd need to explore more and find a way to group them. 
    
</details>

* How many different values are there in the 'Yearly Amount Spent' column? Write a sentence to argue why this is a regression task, rather than a classification task.

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
This is a regression problem due to the amount of unique target values and the fact that they are continuous.
    
</details>

In [ ]:
# %load answers/04_Regression_Loyalty_Memberships/loyalty_preliminary.py

## 1. Correlation

🔔 *Correlation - a statistic that measures the degree to which two variables move in relation to each other.*

We will try to learn a model that predicts the yearly amount spent by a customer. However, what if we are able to predict that already from one variable alone? Or what if some variables are not relevant for predicting the yearly amount spent? 

Let's investigate whether any of the other variables correlate with the yearly amount spent.

### Visual inspection

Using pairplot we can visualise the relationship between the numerical columns.

In [ ]:
sns.pairplot(customers);

From the plots, ‘Length of Membership’ and ‘Time in App’ appear to be the variables that have the most correlation with the dependent variable.

### Correlation matrix

We can also compute the actual pairwise correlation of the columns.

In [ ]:
customers.corr(numeric_only=True)
# customers.corr()  # for pandas < 1.5.0

### Heatmap

This correlation matrix can be visualised with a heatmap. 

In [ ]:
sns.heatmap(customers.corr(numeric_only=True), linewidth=0.5, annot=True);

Along with the variables mentioned earlier, we can see that Avg. Session Length may also be informative for predicting the dependent variable.

## 2. Build a Linear Regression Model </mark>

### <mark>Exercise: Create Predictor variables 'X' and Target Variable 'y'</mark>

For the time being, let’s form our feature matrix using the variables that appear to have a high degree of correlation with the dependent variable.

Create the X and y variables. X should have the variables: `time on app` and `length of membership`.


In [ ]:
# %load answers/04_Regression_Loyalty_Memberships/create-x-y.py

### <mark>Exercise: Split the data into a training set and a testing set.</mark>

We will train our model on the training set and then use the test set to evaluate the model.

Reserve 30% of the data for testing and set a random seed of 100.

In [ ]:
# %load answers/04_Regression_Loyalty_Memberships/train-test-split.py

### <mark>Exercise: Import the model</mark>

Look up how to import a simple [linear regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)

Import the model and then instantiate it.

In [ ]:
# %load answers/04_Regression_Loyalty_Memberships/instantiate.py

### <mark>Exercise: Train the model</mark>

Fit the model to the train set.

In [ ]:
# %load answers/04_Regression_Loyalty_Memberships/train.py

## 3. Intercept and Coefficients

### Investigate the model's intercept and coefficients.

🔔 What are they?

```python
# Print the intercept
print("Intercept", model.intercept_)

# Print the coefficients
coeff_df = pd.DataFrame(model.coef_, X.columns, columns=['Coefficient'])
print(coeff_df)
```

*TAKE CARE: Make sure you replace `model` if you have instantiated your model with a different name!*

### Interpreting the model (inference)

a) What does the intercept represent in this context?

b) How do the coefficients relate to the predictions of the model?

c) Can you conclude from the coefficients of the predictors that one of them is more predictive than the other?

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
a) The intercept represents the value we would predict, if both predictor variables `time on app` and `length of membership` are zero. So, we can see this as a baseline value.
    
b) The coefficients represent the estimated prediction of "yearly amount spent" based on the following formula:  `yearly amount spent` = 37.14 * `Time on App` + 63.53 * `Length of Membership` - 172.77

c) No, we need to be careful when interpreting these coefficients. Their magnitudes alone are not comparable as the `time on app` is on a different scale than `length of membership`. If all variables were on the same scale we could use the coefficients to compare how predictive each predictor is. But, even then, we would need to be careful about drawing causal conclusions.

</details>

## 4. Model Predictions</mark>

### <mark>Exercise: Use your model to make a prediction on the test set.



In [ ]:
# %load answers/04_Regression_Loyalty_Memberships/predictions.py

## 5. Evaluating Predictions</mark>

We want to evaluate whether we have made good predictions or not.

a) Why would accuracy not be a good metric here?

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
Because with accuracy, we would see how many targets we managed to get **exactly** right. However, we are rarely going to ever be precise with a continuous target - but that doesn't mean we're not really close!
    
</details>

### Visual inspection

b) **Make a scatter plot** that plots the test set against your predictions.

```python
plt.scatter(y_test, y_pred)
plt.xlabel("Test")
plt.ylabel("Predictions")
```

If your linear regression was really accurate what would you expect to see?
How does yours compare?

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
We would expect to see every point to be on a line (so that the predicted value is exactly the true value). We can see in our plot that the predicted and true values are close (thus having someone of a linear trend), but not perfect.
    
</details>

### Regression Metrics

Rather than trying to interpret our graph, we can also calculate some metrics.

#### **Coefficient of Determination**, denoted $R^2$ 
We can use `.score()` method here that, when you apply this method to a linear regression model, returns $R^2$.

Find the coefficient of determination of your model using the following:

```python
model.score(X_test, y_test)
```

🔔 The **Coefficient of Determination**, or $R^2$ score, is a **goodness-of-fit** measurement for regression models. 

It is a measure of the percentage of variance in the target which the features explain collectively. 

Accordingly, the $R^2$ score indicates the strength of the relationship between your model and the target on a scale between **0 – 100%**.

#### Error metrics
🔔 Instead of $R^2$ you could also calculate the difference between each predicted value and the actual value (or the distance from the line to the point). This difference is called a **residual**. 

<img src="images/04_Regression_Loyalty/residuals.png" width="400" align='left'></img>

But how do you get a single metric out of all these residuals? There are several options.

d.) One option could be to take the average value of all residuals (Mean Error). Why would that not be a good idea?

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
The problem with just calculating the mean error is that some predictions are lower than the actual value (represented as positive values) and some are higher (represented as negative values). These errors would then cancel each other out, if we summed them up. 
    
So, at the very least we should calculate the Mean Absolute Error (MAE). In practice, however, the Mean Squared Error (MSE) is more common. This is because the linear regression process tries to minimize the MSE (and not the MAE), because it has better mathematical properties (e.g., being continuously differentiable).    
</details>

🔔 The following metrics are used most:

**Mean Absolute Error (MAE)**

MAE is the sum of absolute differences between our target and predicted variables. So it measures the average magnitude of errors in a set of predictions, without considering their directions.


<img src="images/04_Regression_Loyalty/MAE.png" width="400" align='left'></img>

**Mean Squared Error (MSE)**

The mean squared error tells you how close a regression line is to a set of points. It does this by taking the distances from the points to the regression line (these distances are the “errors”) and squaring them. The squaring is necessary to remove any negative signs.

<img src="images/04_Regression_Loyalty/MSE.png" width="400" align='left'></img>

**Root Mean Squared Error (RMSE)**


The RMSE is the square root of the variance of the residuals. It indicates the absolute fit of the model to the data–how close the observed data points are to the model's predicted values. 


<img src="images/04_Regression_Loyalty/RMSE.png" width="400" align='left'></img>


Import metrics from sklearn and then calulate the following:

e) Mean Absolute Error (MAE), Mean Squared Error (MSE), Root Mean Squared Error (RMSE)

You can find these in `sklearn.metrics`: 
- `mean_absolute_error`
- `mean_squared_error`
- `root_mean_squared_error`

In [ ]:
# %load answers/04_Regression_Loyalty_Memberships/mae-mse.py

f) What is the difference between the metrics, and what will MSE/RMSE be more sensitive to than MAE?

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
  
The MSE/RMSE calculate the mean of the squared errors while MAE just takes absolute values. Since we square the errors, MSE/RMSE will be more sensitive to extreme outliers).
</details>

## 6. Overfitting</mark>

By evaluating predictions on the data we trained on, we can check for underfitting and overfitting.

a) Make predictions on the train set and compute the same metrics as before (MAE, MSE & RMSE).

b) What would you expect to see if we were overfitting?

```python
# get predictions for the train data
train_pred = model.predict(X_train)

# find the MAE, MSE and RMSE for the train and test data
print('Train MAE:', round(metrics.mean_absolute_error(y_train, train_pred),4))
print('Train MSE:', round(metrics.mean_squared_error(y_train, train_pred),4))
print('Train RMSE:', round(np.sqrt(metrics.mean_squared_error(y_train, train_pred)),4))

print('Test MAE:', round(metrics.mean_absolute_error(y_test, y_pred),4))
print('Test MSE:', round(metrics.mean_squared_error(y_test, y_pred),4))
print('Test RMSE:', round(np.sqrt(metrics.mean_squared_error(y_test, y_pred)),4))
```

<details>
    
  <summary><span style="color:blue">Show answer</span></summary>
If we were overfitting, we would expect to see our training error to be smaller than the test error, which would mean our model fitted to closely to the training observations but can not generalize well enough to fit the test observations closely.

</details>

### <mark> Exercise: Incorporating more features </mark>

Remember we had left out a variable which had lesser degree of positive correlation? Add that variable (`avg. session length`) and see if it improves the model.

Compare performance using the same metrics as before (MAE, MSE & RMSE) when this extra information is included in the feature matrix.

In [ ]:
# %load answers/04_Regression_Loyalty_Memberships/extra-feature.py

# Conclusion

We've now seen an example of a regression model and seen how the steps for building this follow that of what we saw in the classification examples. However, you also have learned why there is a need for different metrics, and how you can use regression metrics to evaluate your model.